In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.moirai.circuitlens import CircuitLensMoirai
from tsfm_lens.utils import apply_custom_style, load_dyst_samples, set_seed


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")


In [ ]:
DEFAULT_CONTEXT_LENGTH = 4000

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
STOR_DIR = os.getenv("STOR", "/work")
DATA_DIR = os.path.join(STOR_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_moirai", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
# Load the pre-trained Moirai model
model_id = "Salesforce/moirai-1.1-R-base"
context_length = DEFAULT_CONTEXT_LENGTH

pipeline = CircuitLensMoirai(
    model_name=model_id,
    context_length=DEFAULT_CONTEXT_LENGTH,
    prediction_length=1,  # NOTE: this is arbitrary, just a placeholder
    # patch_size="auto",
    patch_size=32,
    num_samples=100,
    target_dim=1,
    device=device,
)


In [ ]:
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")


In [ ]:
model_param_memory_mb = sum(p.numel() * p.element_size() for p in pipeline.model.parameters()) / (1024 * 1024)
print(f"Model parameter memory: {model_param_memory_mb:.2f} MB")


### Load Data

In [ ]:
split_name = "gift-eval"
subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Thomas"
    system_name_pretty = system_name

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    sample_idx = 0
    selected_dim = [0, 1, 2]

    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :])
    print(dyst_coords.shape)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]
    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"context shape: {context.shape}, future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    from itertools import islice

    split_dir = os.path.join(DATA_DIR, split_name)
    # system_name = "ett1/15T"
    system_name = "LOOP_SEATTLE/H"
    to_univariate, term = False, "medium"
    selected_sample_idx, selected_dim = 1, [0, 1, 2]

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)

    system_name_pretty = f"{system_name.replace('/', '_')}_dim{selected_dim}_sample{selected_sample_idx}"

    prediction_length = dataset.test_data.prediction_length
    print(f"Prediction length: {prediction_length}, Windows: {dataset.windows}")

    # Get the selected sample efficiently
    context_entry = next(islice(dataset.test_data.input, selected_sample_idx, None))
    future_entry = next(islice(dataset.test_data.label, selected_sample_idx, None))
    print(f"Item: {context_entry['item_id']}, Freq: {context_entry['freq']}")

    # Extract and select dimensions
    context_target = context_entry["target"]
    future_target = future_entry["target"]
    if context_target.ndim == 1:
        context_target, future_target = context_target[None, :], future_target[None, :]
    else:
        context_target, future_target = context_target[selected_dim], future_target[selected_dim]

    # Limit context to last DEFAULT_CONTEXT_LENGTH points and adjust start time
    orig_context_len = context_target.shape[1]
    truncate_offset = max(0, orig_context_len - DEFAULT_CONTEXT_LENGTH)
    context_target = context_target[:, -DEFAULT_CONTEXT_LENGTH:]
    context_start = context_entry["start"] + truncate_offset
    print(f"Context: {context_target.shape}, Future: {future_target.shape}")

    # Plot
    num_variates = context_target.shape[0]
    fig, axes = plt.subplots(num_variates, 1, figsize=(5, 2 * num_variates), squeeze=False)
    for dim, ax in enumerate(axes.flat):
        to_pandas({"target": context_target[dim], "start": context_start}).plot(
            color="black", linewidth=1, ax=ax, label="Context"
        )
        to_pandas({"target": future_target[dim], "start": future_entry["start"]}).plot(
            color="tab:blue", linewidth=1, ax=ax, label="Ground Truth"
        )
        ax.grid(which="both")
        ax.set_title(f"Dim {dim}")
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Convert to torch tensors
    context = torch.tensor(context_target)
    future_vals = torch.tensor(future_target)
    num_nans = np.isnan(context_target).sum() + np.isnan(future_target).sum()
    print(f"Context: {context.shape}, Future: {future_vals.shape}, NaNs: {num_nans}")


### Ablations

In [ ]:
import json
from itertools import groupby

ASSETS_DIR = os.path.join("../../", "assets")

heads_to_ablate = json.load(
    open(os.path.join(ASSETS_DIR, f"{pipeline.__class__.__name__}_ablate_for_heads_at_1pp.json"))
)

# chosen_layers = list(range(pipeline.num_layers))
chosen_layers = [3, 4, 5, 6]

heads_to_ablate = [config for config in heads_to_ablate if config[0] in chosen_layers]

num_heads_per_layer_to_skip = 3
layers_to_keep_at_heads1pp = []
# layers_to_keep_at_heads1pp = []

if num_heads_per_layer_to_skip == 0:
    layers_to_keep_at_heads1pp = chosen_layers
# heads_to_ablate is a list of lists each [layer, head] (should be a tuple, but we can keep it a list for now)
# for each layer, we skip the last head that has that layer as the first element of the tuple
# so we skip the last head for layer 0, the last head for layer 1, etc.

# Group heads by layer and remove the last num_heads_per_layer_to_skip heads from each layer
# except for layers in layers_to_keep_at_heads1pp

heads_to_ablate = [
    config
    for layer, group in groupby(sorted(heads_to_ablate, key=lambda x: x[0]), key=lambda x: x[0])
    for config in (list(group) if layer in layers_to_keep_at_heads1pp else list(group)[:-num_heads_per_layer_to_skip])
]


In [ ]:
chosen_layers_mlp = []
print(f"layers chosen for MLP ablation: {chosen_layers_mlp}")

In [ ]:
pipeline.remove_all_hooks()

# print number of heads ablated per layer
for layer in chosen_layers:
    heads_in_layer = [config[1] for config in heads_to_ablate if config[0] == layer]
    print(f"Layer {layer}: {len(heads_in_layer)} heads")

n_heads_ablated = len(heads_to_ablate)
print(f"{n_heads_ablated} heads_to_ablate: {heads_to_ablate}")

ablations_summary_str = f"ablate_{n_heads_ablated}_heads"

print(f"ablations_summary_str: {ablations_summary_str}")

pipeline.add_ablation_hooks_explicit(
    ablations_types=["head", "mlp"],  # type: ignore
    layers_to_ablate_mlp=chosen_layers_mlp,
    heads_to_ablate=heads_to_ablate,
)

layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

ablations_summary_str_suffix = ""
if layers_without_heads and layers_without_mlps:
    # if layers_without_heads != layers_without_mlps:
    #     raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = (
        f"Zero-Ablation: Heads in Layers {layers_without_heads}, MLPs in Layers {layers_without_mlps}"
    )
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}-mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix
print(ablations_summary_str)
print(ablations_summary_str_title)


In [ ]:
layers_without_heads = list(pipeline.head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

ablations_summary_str_suffix = ""
if layers_without_heads and layers_without_mlps:
    # if layers_without_heads != layers_without_mlps:
    #     raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
    ablations_summary_str_title = (
        f"Zero-Ablation: Heads in Layers {layers_without_heads}, MLPs in Layers {layers_without_mlps}"
    )
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}-mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str_suffix = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str_suffix = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str_suffix = ablations_summary_str_title = None

ablations_summary_str += "_" + ablations_summary_str_suffix
print(ablations_summary_str)
print(ablations_summary_str_title)

In [ ]:
save_fname = (
    f"predictions_{system_name_pretty}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name_pretty}"
)

### Predict and Get Outputs

In [ ]:
rseed = 123
num_samples = 20
set_seed(rseed)

pred = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,
)


In [ ]:
pred.shape

### Plot Predictions with Ablations

In [ ]:
plot_samples = False

In [ ]:
context_np = context.cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.cpu().numpy()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)

print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")


In [ ]:
preds = pred.detach().cpu().numpy()
if preds.ndim == 2:
    preds = pred[None, ...]
num_variates, num_samples, horizon = preds.shape

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(context_timesteps[-512:], context_np[dim][-512:], "k-", linewidth=1, label="Context")
    ax.plot(future_timesteps, future_vals_np[dim], "k--", label="Ground Truth", linewidth=1, zorder=10)
    if plot_samples:
        for i in range(num_samples):
            ax.plot(future_timesteps, preds[dim, i], linewidth=1, color=DEFAULT_COLORS[i], alpha=0.2)
    ax.plot(
        future_timesteps,
        np.median(preds, axis=1)[dim],
        color="tab:blue",
        linewidth=1,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)


plt.tight_layout()


### Plot Original Predictions

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(
    context,
    prediction_length=prediction_length,
    num_samples=num_samples,
)


In [ ]:
num_variates, num_samples, horizon = pred_original.shape
preds_original = pred_original.detach().cpu().numpy()

fig, axs = plt.subplots(num_variates, 1, figsize=(6, 2 * num_variates))
axs = [axs] if num_variates == 1 else axs

for dim, ax in enumerate(axs):
    ax.plot(
        context_timesteps[-min(512, len(context_timesteps)) :],
        context_np[dim][-min(512, len(context_timesteps)) :],
        "k-",
        linewidth=1,
        label="Context",
    )
    ax.plot(future_timesteps, future_vals_np[dim], "k--", label="Ground Truth", linewidth=1, zorder=10)
    if plot_samples:
        for i in range(num_samples):
            ax.plot(
                future_timesteps,
                preds_original[dim, i],
                linewidth=1,
                color=DEFAULT_COLORS[i % len(DEFAULT_COLORS)],
                alpha=0.2,
            )
    ax.plot(
        future_timesteps,
        np.median(preds_original, axis=1)[dim],
        color="tab:blue",
        linewidth=1,
        label="Median",
        zorder=11,
    )
    ax.set_xlabel("Timestep", fontweight="bold")
    ax.set_ylabel(f"Dimension {dim}", fontweight="bold")
    ax.legend(loc="lower left", frameon=True)

plt.tight_layout()


### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=1)
med_orig = np.median(preds_original, axis=1)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")


In [ ]:
pred_horizon_lst = [64]
for pred_horizon in pred_horizon_lst:
    for dim in range(med_pred.shape[0]):
        print(f"Computing metrics for prediction horizon {pred_horizon} and dimension {dim}")
        metrics_combined = compute_metrics(med_orig[dim, :pred_horizon], med_pred[dim, :pred_horizon], verbose=True)
        spearman_corr = metrics_combined["spearman"]
        print(f"Spearman correlation: {spearman_corr}")
